In [ ]:
import seaborn as sns
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from astropy.time import Time
from LINZ.CORS_Timeseries import TimeseriesList, CoordApiTimeseries
import os 
from statsmodels.tsa.stattools import acf
import matplotlib.pyplot as plt
import re
from typing import Dict

In [ ]:
# Define time range
start = datetime(2025, 4, 10)
end = datetime(2025, 10, 21)

In [ ]:
# Define station groups
PNcodes=['AUCK','BLUF','CHTI','CORM','DNVK','DUND','GISB','GLDB','HAAS','HAMT','HAST','HIKB','HOKI','KAIK','KTIA','LEXA','LKTA','MAHO','MAVL','METH','MQZG','MTJO','NLSN','NPLY','PYGR','TAUP','TRNG',
         'VGMT','WAIM','WANG','WARK','WEST','WGTN','WHKT','WHNG','WRPA']

In [ ]:
# Calculate crd_epoch
crd_epoch = end + (start - end) / 2
astropy_time_object = Time(crd_epoch, format='datetime')
crd_epoch_decimal_year = astropy_time_object.decimalyear

In [ ]:
# --- Clear, human-readable labels for solution types ---
SOLUTION_LABEL_MAP = {
    'd20f_54_code_A_G': 'GPS',
    'd20f_54_code_AGE': 'GPS+Galileo',
    'd20f_54_code_A':   'GPS+Galileo+GLONASS',
    'd20f_54_mgex_A':   'GPS+Galileo+BeiDou'
}

# Initialise a dictionary to store DataFrames for each solution type
daily_avg_dfs = {
    # Keep keys as your original solution types for API calls and grouping
    'd20f_54_code_A_G': {},  # GPS only
    'd20f_54_code_AGE': {},  # GPS + Galileo
    'd20f_54_code_A':   {},  # GPS + Galileo + GLONASS
    'd20f_54_mgex_A':   {},  # GPS + Galileo + BeiDou
}

# Loop through PNcodes and solution types
for code in PNcodes:
    for sol_type in daily_avg_dfs.keys():
        try:
            print(f"Processing station: {code} with solution type: {sol_type}")
            ts = CoordApiTimeseries(code, solutiontype=sol_type, after=start, before=end)
            dates, xyz = ts.withoutOutliers().getObs(enu=False)

            df_obs = pd.DataFrame(xyz, columns=['x', 'y', 'z'], index=dates)
            df_obs['date'] = df_obs.index.strftime('%Y-%m-%d')
            df_obs['station'] = code

            # Apply the human-readable label
            df_obs['solutiontype'] = SOLUTION_LABEL_MAP.get(sol_type, sol_type)

            daily_avg_dfs[sol_type][code] = df_obs

        except Exception as e:
            print(f"Failed to process station {code} with solution type {sol_type}. Error: {e}")

# Combine all DataFrames into one
PN_data = pd.concat(
    [pd.concat(dfs.values(), axis=0) for dfs in daily_avg_dfs.values()],
    axis=0
)

# Sort alphabetically by station, date, and solution type
PN_data.sort_values(by=["station", "date", "solutiontype"], inplace=True)

# Print confirmation
print("Download complete ✅")
print("PN_data DataFrame created with shape:", PN_data.shape)
print("Columns:", PN_data.columns.tolist())
print("Solution types present:", sorted(PN_data['solutiontype'].unique()))

In [ ]:
# Group PN_data by station and solutiontype
group_dataframes = {
    f"{station}_{sol_type}": group
    for (station, sol_type), group in PN_data.groupby(["station", "solutiontype"])
}

In [ ]:
# Define expected converted DataFrame names
converted_columns = [
    'x', 'y', 'z', 'date', 'station',
    'nztm_easting', 'nztm_northing', 'nzvd2016_elev'
]

# Initialize empty dictionary with expected keys
empty_converted_dfs = {
    f"{name}_converted": pd.DataFrame(columns=converted_columns)
    for name in group_dataframes.keys()
}

In [ ]:
converted_group_dataframes = {}

def convert_coordinates(input_crds, crd_epoch_decimal_year):
    occ_api = "https://www.geodesy.linz.govt.nz/api/conversions/v1/convert-to"
    coordlist = {
        "crs": "LINZ:ITRF2020_XYZ",
        "coordinateOrder": ["geocentricX", "geocentricY", "geocentricZ"],
        "coordinateEpoch": crd_epoch_decimal_year,
        "coordinates": input_crds
    }
    params = {"crs": "LINZ:NZTM/NZVD2016"}

    response = requests.post(occ_api, params=params, json=coordlist)
    if response.status_code == 200:
        converted = response.json()
        return converted['coordinateList']['coordinates']
    else:
        print(f"❌ API conversion failed with status {response.status_code}")
        print(f"🔍 Error details: {response.text}")
        return []


In [ ]:
for name, original_df in group_dataframes.items():
    print(f"\n🔄 Processing DataFrame: {name}")

    # Always start with a fresh copy
    df = original_df.copy(deep=True)

    # Identify and print NaN values
    nan_values = df[df.isna().any(axis=1)]
    if not nan_values.empty:
        print(f" - Found {len(nan_values)} NaN rows")

    # Identify and print Infinity values
    inf_values = df[(df == np.inf) | (df == -np.inf)].dropna(how='all')
    if not inf_values.empty:
        print(f" - Found {len(inf_values)} Inf rows")

    # Filter out invalid rows
    df_filtered = df.dropna(subset=['x', 'y', 'z']).copy()
    df_filtered = df_filtered[
        (df_filtered['x'] != np.inf) & (df_filtered['x'] != -np.inf) &
        (df_filtered['y'] != np.inf) & (df_filtered['y'] != -np.inf) &
        (df_filtered['z'] != np.inf) & (df_filtered['z'] != -np.inf)
    ]

    if df_filtered.empty:
        print(f"⚠️ Skipping {name} — no valid coordinates after filtering.")
        continue

    # Prepare coordinates
    input_crds = df_filtered[['x', 'y', 'z']].values.tolist()

    # Convert coordinates
    converted_coords = convert_coordinates(input_crds, crd_epoch_decimal_year)

    # Add converted coordinates to DataFrame
    if converted_coords:
        df_filtered[['nztm_easting', 'nztm_northing', 'nzvd2016_elev']] = pd.DataFrame(
            converted_coords, index=df_filtered.index
        )

        # Store in the pre-initialised DataFrame
        converted_name = f"{name}_converted"
        if converted_name in empty_converted_dfs:
            empty_converted_dfs[converted_name] = df_filtered
            print(f"✅ {converted_name} filled and updated.")
        else:
            print(f"⚠️ {converted_name} not found in empty_converted_dfs. Skipping storage.")
    else:
        print(f"❌ Conversion failed for {name}. No coordinates added.")

In [ ]:
# Rename for clarity — this is now the final dictionary of converted DataFrames
converted_group_dataframes = empty_converted_dfs
del empty_converted_dfs  # Optional: clean up the old name if no longer needed

In [ ]:
# --- Imports ---
import re
import random
from typing import Dict, Sequence, List, Optional

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# Helper: reproducible random sampling of stations (standard library)
def sample_stations(
    stations: Sequence[str],
    k: int,
    seed: int | None = None,
    verbose: bool = False,
    tag: str = ""
) -> List[str]:
    stations = sorted(set(stations))
    if k is None or len(stations) <= k:
        if verbose:
            print(f"[{tag}] Sampling request k={k}; using {len(stations)} stations.", flush=True)
        return stations
    rng = random.Random(seed)
    chosen = sorted(rng.sample(stations, k))
    if verbose:
        print(f"[{tag}] Randomly selected {k} station(s) (seed={seed}).", flush=True)
    return chosen


def plot_available_solutions(
    converted_group_dataframes: Dict[str, pd.DataFrame],
    window_days: int = 30,
    min_solutions_per_station: int = 1,  # set to 2 if you only want comparisons
    verbose: bool = True
):
    if verbose:
        print(">>> Entered plot_available_solutions", flush=True)
        print(f"Dict received: {len(converted_group_dataframes)} keys", flush=True)
        # show a couple of keys immediately
        print("Sample keys:", list(converted_group_dataframes.keys())[:5], flush=True)

    # 1) Discover networks/solutions from keys
    key_re = re.compile(r"^(?P<network>[^_]+)_(?P<solution>.+?)_converted$")  # <-- ensure < and > !

    ns_map: Dict[str, Dict[str, pd.DataFrame]] = {}
    unmatched = []

    for key, df in converted_group_dataframes.items():
        m = key_re.match(key)
        if not m:
            unmatched.append(key)
            continue
        net = m.group("network")
        sol = m.group("solution")
        ns_map.setdefault(net, {})[sol] = df.copy()

    if verbose:
        print("=== DISCOVERY ===", flush=True)
        print("Networks found:", sorted(ns_map.keys()), flush=True)
        for net, sols in sorted(ns_map.items()):
            print(f"  [{net}] {len(sols)} solution types -> {sorted(sols.keys())}", flush=True)
        if unmatched:
            print(f"Unmatched keys (ignored): {len(unmatched)} (sample: {unmatched[:5]})", flush=True)

    if not ns_map:
        print("No keys matched pattern {network}_{solution}_converted — nothing to plot.", flush=True)
        return

    # 2) Process each network
    for network, sols_dict in sorted(ns_map.items()):
        if verbose:
            print(f"\n=== NETWORK: {network} ===", flush=True)

        # Clean / normalise frames
        for sol, df in list(sols_dict.items()):
            required_cols = ("date", "station", "nzvd2016_elev")
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                if verbose:
                    print(f"[{network}/{sol}] Missing columns {missing}; dropping this solution.", flush=True)
                sols_dict.pop(sol)
                continue

            # Parse dates and tidy stations
            before_rows = len(df)
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df.dropna(subset=['date'], inplace=True)
            df['station'] = df['station'].astype(str).str.strip()
            after_rows = len(df)
            sols_dict[sol] = df

            if verbose:
                print(f"[{network}/{sol}] Rows: {before_rows} -> {after_rows} after cleaning", flush=True)

        if not sols_dict:
            if verbose:
                print(f"[{network}] No usable solutions after cleaning; skipping.", flush=True)
            continue

        # Union of stations
        stations_union = sorted(set().union(*[set(df['station']) for df in sols_dict.values()]))
        if verbose:
            print(f"[{network}] Stations (union across solutions): {len(stations_union)}", flush=True)
            if len(stations_union) == 0:
                print(f"[{network}] No stations available; skipping.", flush=True)

        # --- OPTIONAL: enable seeded random sampling per network (comment/uncomment) ---
        # stations_union = sample_stations(stations_union, k=10, seed=42, verbose=verbose, tag=network)

        # Stable palette per network based on solutions available
        solution_names = sorted(sols_dict.keys())
        palette = dict(zip(solution_names, sns.color_palette("colorblind", n_colors=len(solution_names))))

        # 3) Per-station plotting
        for station_name in stations_union:
            station_frames = {}
            for sol, df in sols_dict.items():
                d = (
                    df.loc[df['station'] == station_name]
                      .sort_values('date')
                      .set_index('date')
                      .copy()
                )
                if d.empty:
                    if verbose:
                        print(f"[{network}/{station_name}] '{sol}' has no rows; continuing.", flush=True)
                    continue

                # Metrics
                d['running_mean'] = d['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
                d['detrended_mm'] = (d['nzvd2016_elev'] - d['running_mean']) * 1000
                d['elevation'] = d['nzvd2016_elev']
                d['solutiontype'] = sol
                station_frames[sol] = d

            # Require at least N solutions with rows to plot (default 1 = permissive)
            if len(station_frames) < min_solutions_per_station:
                if verbose:
                    print(f"[{network}/{station_name}] Skipping: only {len(station_frames)} solution(s) with rows "
                          f"(need ≥ {min_solutions_per_station}).", flush=True)
                continue

            combined_df = pd.concat(station_frames.values()).reset_index()

            if verbose:
                print(f"[{network}/{station_name}] Plotting {len(combined_df)} rows across "
                      f"{len(station_frames)} solution(s): {sorted(station_frames.keys())}", flush=True)

            # Plot
            sns.set(style="whitegrid")
            fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

            sns.lineplot(
                data=combined_df, x='date', y='elevation', hue='solutiontype',
                ax=ax[0], alpha=0.6, palette=palette, linewidth=1, linestyle='--',
                legend=False
            )
            sns.lineplot(
                data=combined_df, x='date', y='running_mean', hue='solutiontype',
                ax=ax[0], linewidth=2, palette=palette
            )
            ax[0].set_title(f'Elevation and Running Mean — {network} / {station_name}')
            ax[0].set_ylabel('Elevation (m)')
            ax[0].legend(title='Solution Type')
            ax[0].grid(True)

            sns.lineplot(
                data=combined_df, x='date', y='detrended_mm', hue='solutiontype',
                ax=ax[1], palette=palette, linewidth=1
            )
            ax[1].axhline(0, color='grey', linestyle='--', linewidth=1)
            ax[1].set_title(f'Detrended Residuals — {network} / {station_name}')
            ax[1].set_xlabel('Date')
            ax[1].set_ylabel('Residuals (mm)')
            ax[1].legend(title='Solution Type')
            ax[1].grid(True)

            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
            plt.pause(0.001)  # nudge for some backends

    if verbose:
        print(">>> Finished plot_available_solutions", flush=True)

In [ ]:
# --- Imports ---
import re
from typing import Dict

def plot_available_solutions(
    converted_group_dataframes: Dict[str, pd.DataFrame],
    window_days: int = 30,
    min_solutions_per_station: int = 1,  # set to 2 if you only want comparisons
    verbose: bool = True
):
    """
    Discover networks and solution types from keys:
        {network}_{solution}_converted

    For each network and station, plot *whatever* solutions have data for that station.
    Displays plots only (no saving).
    """

    # ---------- 1) Discover networks/solutions from keys ----------
    key_re = re.compile(r"^(?P<network>[^_]+)_(?P<solution>.+?)_converted$")

    ns_map: Dict[str, Dict[str, pd.DataFrame]] = {}
    unmatched = []

    for key, df in converted_group_dataframes.items():
        m = key_re.match(key)
        if not m:
            unmatched.append(key)
            continue
        net = m.group("network")
        sol = m.group("solution")
        ns_map.setdefault(net, {})[sol] = df.copy()

    if verbose:
        print("=== DISCOVERY ===")
        print(f"Total dict keys: {len(converted_group_dataframes)}")
        print("Sample keys:", list(converted_group_dataframes.keys())[:10])
        print("Networks found:", sorted(ns_map.keys()))
        for net, sols in sorted(ns_map.items()):
            print(f"  [{net}] {len(sols)} solution types -> {sorted(sols.keys())}")
        if unmatched:
            print(f"Unmatched keys (ignored): {len(unmatched)} (sample: {unmatched[:5]})")

    if not ns_map:
        print("No keys matched pattern {network}_{solution}_converted — nothing to plot.")
        return

    # ---------- 2) Process each network ----------
    for network, sols_dict in sorted(ns_map.items()):
        if verbose:
            print(f"\n=== NETWORK: {network} ===")

        # Clean / normalise frames
        for sol, df in list(sols_dict.items()):
            # Require key columns
            required_cols = ("date", "station", "nzvd2016_elev")
            missing = [c for c in required_cols if c not in df.columns]
            if missing:
                if verbose:
                    print(f"[{network}/{sol}] Missing columns {missing}; dropping this solution.")
                sols_dict.pop(sol)
                continue

            # Parse dates and tidy stations
            df['date'] = pd.to_datetime(df['date'], errors='coerce')
            df.dropna(subset=['date'], inplace=True)
            df['station'] = df['station'].astype(str).str.strip()
            sols_dict[sol] = df

        if not sols_dict:
            if verbose:
                print(f"[{network}] No usable solutions after cleaning; skipping.")
            continue

        # Union of stations (plot whatever has data for each station)
        stations_union = sorted(set().union(*[set(df['station']) for df in sols_dict.values()]))

        if verbose:
            print(f"[{network}] Stations (union across solutions): {len(stations_union)}")

        # Stable palette per network based on the solutions available there
        solution_names = sorted(sols_dict.keys())
        palette = dict(zip(solution_names, sns.color_palette("colorblind", n_colors=len(solution_names))))

        # ---------- 3) Per‑station plotting ----------
        for station_name in stations_union:
            station_frames = {}
            for sol, df in sols_dict.items():
                d = (
                    df.loc[df['station'] == station_name]
                      .sort_values('date')
                      .set_index('date')
                      .copy()
                )
                if d.empty:
                    if verbose:
                        print(f"[{network}/{station_name}] '{sol}' has no rows; continuing.")
                    continue

                # Metrics
                d['running_mean'] = d['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
                d['detrended_mm'] = (d['nzvd2016_elev'] - d['running_mean']) * 1000
                d['elevation'] = d['nzvd2016_elev']
                d['solutiontype'] = sol
                station_frames[sol] = d

            # Require at least N solutions with rows to plot (default 1 = permissive)
            if len(station_frames) < min_solutions_per_station:
                if verbose:
                    print(f"[{network}/{station_name}] Skipping: only {len(station_frames)} solution(s) with rows "
                          f"(need ≥ {min_solutions_per_station}).")
                continue

            combined_df = pd.concat(station_frames.values()).reset_index()

            if verbose:
                print(f"[{network}/{station_name}] Plotting {len(combined_df)} rows across "
                      f"{len(station_frames)} solution(s): {sorted(station_frames.keys())}")

            # --- Plot ---
            sns.set(style="whitegrid")
            fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

            # Elevation (exclude from legend)
            sns.lineplot(
                data=combined_df, x='date', y='elevation', hue='solutiontype',
                ax=ax[0], alpha=0.6, palette=palette, linewidth=1, linestyle='--',
                legend=False                                 
            )
            
            # Running mean (legend ON)
            sns.lineplot(
                data=combined_df, x='date', y='running_mean', hue='solutiontype',
                ax=ax[0], linewidth=2, palette=palette       
            )
            
            ax[0].set_title(f'Elevation and Running Mean — {network} / {station_name}')
            ax[0].set_ylabel('Elevation (m)')
            ax[0].legend(title='Solution Type')              
            ax[0].grid(True)

            # Detrended residuals
            sns.lineplot(
                data=combined_df, x='date', y='detrended_mm', hue='solutiontype',
                ax=ax[1], palette=palette, linewidth=1
            )
            ax[1].axhline(0, color='grey', linestyle='--', linewidth=1)
            ax[1].set_title(f'Detrended Residuals — {network} / {station_name}')
            ax[1].set_xlabel('Date')
            ax[1].set_ylabel('Residuals (mm)')
            ax[1].legend(title='Solution Type')
            ax[1].grid(True)

            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

# --- Call the function ---
# Adjust min_solutions_per_station to 2 if you only want stations where at least two solutions have data
plot_available_solutions(
    converted_group_dataframes,
    window_days=30,
    min_solutions_per_station=1,
    verbose=True
)

In [ ]:
station_metrics = []

for key, df in converted_group_dataframes.items():
    if df.empty or 'nzvd2016_elev' not in df.columns or 'date' not in df.columns or 'station' not in df.columns:
        continue

    df['date'] = pd.to_datetime(df['date'])

    for station_name, station_df in df.groupby('station'):
        station_df = station_df.sort_values('date').set_index('date')
        window_days = (station_df.index.max() - station_df.index.min()).days
        station_df['running_mean'] = station_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
        station_df['detrended_mm'] = (station_df['nzvd2016_elev'] - station_df['running_mean']) * 1000

        noise_std = station_df['detrended_mm'].std()

        parts = key.split("_")
        network = parts[0]
        solutiontype = "_".join(parts[1:-1])  # exclude 'converted'

        station_metrics.append({
            'network': network,
            'station': station_name,
            'solutiontype': solutiontype,
            'noise_std_mm': noise_std
        })

metrics_df = pd.DataFrame(station_metrics)


In [ ]:
# Make sure metrics_df is defined before running this
plt.figure(figsize=(18, 8))
sns.set(style="whitegrid")

sns.barplot(
    data=metrics_df,
    x='station',
    y='noise_std_mm',
    hue='solutiontype',
    palette="colorblind"
)

plt.title('GNSS Noise Level by Station and Solution Type Across PositioNZ Network')
plt.xlabel('Station')
plt.ylabel('Noise Level (mm)')
plt.xticks(rotation=90, ha='center')
plt.legend(title='Solution Type')
plt.grid(True, which='major', axis='y', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
from matplotlib.backends.backend_pdf import PdfPages

# metrics_df already created by your code above
dfm = metrics_df.dropna(subset=['noise_std_mm', 'solutiontype', 'network'])

# Order solution types by median noise for readability
order = (dfm.groupby('solutiontype')['noise_std_mm']
           .median()
           .sort_values()
           .index)

sns.set(style="whitegrid")

OUTPUT_PDF = "noise_std_boxplots_A4.pdf"
with PdfPages(OUTPUT_PDF) as pdf:
    # --- Page 1: Overall ---
    fig, ax = plt.subplots(figsize=(11.7, 8.3))  # A4 landscape
    sns.boxplot(
        data=dfm,
        x='solutiontype',
        y='noise_std_mm',
        order=order,
        ax=ax,
        showfliers=True,
        whis=1.5
    )
    # Overlay individual stations to show sampling
    sns.stripplot(
        data=dfm,
        x='solutiontype',
        y='noise_std_mm',
        order=order,
        ax=ax,
        color='0.3',
        size=3,
        jitter=True,
        alpha=0.6
    )
    ax.set_title('Distribution of station noise (std of detrended heights) by solution type')
    ax.set_xlabel('Solution type')
    ax.set_ylabel('Noise standard deviation (mm)')
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # --- Page 2: Facet by network (if multiple networks present) ---
    g = sns.catplot(
        data=dfm, kind='box',
        x='solutiontype', y='noise_std_mm',
        col='network', col_wrap=2, sharey=True,
        height=4, aspect=1.6,
        showfliers=True, whis=1.5
    )
    g.set_axis_labels('Solution type', 'Noise standard deviation (mm)')
    g.set_titles('{col_name}')
    # optional scatter overlay for each facet
    for ax in g.axes.flat:
        sns.stripplot(
            data=dfm[dfm['network'] == ax.get_title()],
            x='solutiontype', y='noise_std_mm',
            order=order, ax=ax,
            color='0.3', size=2.5, jitter=True, alpha=0.6
        )
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    g.fig.set_size_inches(11.7, 8.3)  # ensure A4 page sizing
    pdf.savefig(g.fig)
    plt.close(g.fig)

print(f"Saved printable PDF: {OUTPUT_PDF}")

In [ ]:
# ---- INPUT: metrics_df must exist with ['network','station','solutiontype','noise_std_mm'] ----
dfm = metrics_df.dropna(subset=['station','solutiontype','noise_std_mm']).copy()

# Aggregate: median noise per station × solution type
pivot_med = (dfm
    .groupby(['station','solutiontype'])['noise_std_mm']
    .median()
    .unstack('solutiontype'))

# Optional: keep only stations with enough solution types populated
min_solutions = max(2, int(pivot_med.shape[1] * 0.4))  # e.g., at least 40% coverage
pivot_med = pivot_med.loc[pivot_med.count(axis=1) >= min_solutions]

# Order stations by overall median noise for readability
station_order = pivot_med.median(axis=1).sort_values().index
# Order solution types by overall median noise
solution_order = pivot_med.median(axis=0).sort_values().index

pivot_med = pivot_med.loc[station_order, solution_order]

# Row-normalised: subtract each station’s median (emphasises relative differences across solution types)
pivot_rownorm = pivot_med.sub(pivot_med.median(axis=1), axis=0)

sns.set(style="whitegrid")
OUTPUT_PDF = "heatmaps_noise_std_A4.pdf"
with PdfPages(OUTPUT_PDF) as pdf:
    # --- Page 1: Raw median noise heatmap ---
    fig, ax = plt.subplots(figsize=(11.7, 8.3))  # A4 landscape
    cmap = sns.color_palette("viridis", as_cmap=True)
    sns.heatmap(
        pivot_med,
        ax=ax,
        cmap=cmap,
        cbar_kws={"label": "Median noise std (mm)"},
        linewidths=0.3,
        linecolor='white'
    )
    ax.set_title("Station × Solution Type — Median Noise Standard Deviation (mm)")
    ax.set_xlabel("Solution type")
    ax.set_ylabel("Station")
    # Improve label legibility if many stations
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # --- Page 2: Row‑normalised heatmap (relative differences) ---
    fig, ax = plt.subplots(figsize=(11.7, 8.3))
    # Diverging colormap centred at 0
    vlim = np.nanmax(np.abs(pivot_rownorm.values))
    cmap_div = sns.diverging_palette(220, 20, as_cmap=True)
    sns.heatmap(
        pivot_rownorm,
        ax=ax,
        cmap=cmap_div,
        center=0,
        vmin=-vlim, vmax=vlim,
        cbar_kws={"label": "Deviation from station median (mm)"},
        linewidths=0.3,
        linecolor='white'
    )
    ax.set_title("Station × Solution Type — Row‑normalised (deviation from station median)")
    ax.set_xlabel("Solution type")
    ax.set_ylabel("Station")
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8, rotation=45)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

print(f"Saved printable PDF: {OUTPUT_PDF}")

In [ ]:
records = []
for key, df in converted_group_dataframes.items():
    if df.empty or 'nzvd2016_elev' not in df.columns or 'date' not in df.columns or 'station' not in df.columns:
        continue

    df['date'] = pd.to_datetime(df['date'])
    parts = key.split("_")
    network = parts[0]
    solutiontype = "_".join(parts[1:-1])  # exclude 'converted' if that's the suffix

    for station_name, station_df in df.groupby('station'):
        station_df = station_df.sort_values('date').set_index('date')
        window_days = (station_df.index.max() - station_df.index.min()).days
        if window_days <= 0:
            continue
        station_df['running_mean'] = station_df['nzvd2016_elev'].rolling(f'{window_days}D', min_periods=1).mean()
        station_df['detrended_mm'] = (station_df['nzvd2016_elev'] - station_df['running_mean']) * 1000

        for dt, val in station_df['detrended_mm'].items():
            records.append({
                'network': network,
                'solutiontype': solutiontype,
                'station': station_name,
                'date': dt,
                'detrended_mm': float(val)
            })

detrended_df = pd.DataFrame(records).dropna(subset=['detrended_mm'])

In [ ]:
# ---- INPUT ----
df = detrended_df.copy()  # must have: solutiontype, detrended_mm
df = df.dropna(subset=['solutiontype', 'detrended_mm'])

# Optional: winsorise extreme tails to stabilise visuals (set to 0 to disable)
WINSOR_PCT = 0.0   # e.g., 0.01 for 1% clipping
ROBUST = True      # use robust sigma via MAD; set False to use std

def fit_gaussian(x, robust=True):
    x = np.asarray(x)
    mu = float(np.mean(x))
    if robust:
        med = np.median(x)
        mad = np.median(np.abs(x - med))
        sigma = 1.4826 * mad  # robust sigma ≈ MAD * 1.4826
    else:
        sigma = float(np.std(x, ddof=1))
    return mu, max(sigma, 1e-9)  # avoid zero

# Prepare ordering by dispersion
sol_order = (df.groupby('solutiontype')['detrended_mm']
               .std().sort_values().index)

# Determine common x-limits (symmetrical around 0) for consistent pages
abs_max = (df.groupby('solutiontype')['detrended_mm']
             .apply(lambda s: np.nanpercentile(np.abs(s), 99))).max()
xmax = float(abs_max)
xgrid = np.linspace(-xmax, xmax, 800)

OUTPUT_PDF = "normal_overlays_by_solutiontype_A4.pdf"
sns.set(style="whitegrid")
with PdfPages(OUTPUT_PDF) as pdf:
    for sol in sol_order:
        x = df.loc[df['solutiontype'] == sol, 'detrended_mm'].to_numpy()
        if WINSOR_PCT > 0:
            lo = np.nanpercentile(x, 100*WINSOR_PCT)
            hi = np.nanpercentile(x, 100*(1-WINSOR_PCT))
            x = np.clip(x, lo, hi)
        mu, sigma = fit_gaussian(x, robust=ROBUST)
        # Normal PDF
        normal_pdf = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((xgrid - mu)/sigma)**2)

        fig, ax = plt.subplots(figsize=(11.7, 8.3))  # A4 landscape
        # Histogram (density)
        ax.hist(x, bins=60, density=True, color='#9ecae1', edgecolor='#1f77b4', alpha=0.6, label='Histogram (density)')
        # KDE
        sns.kdeplot(x, ax=ax, color='#2ca02c', linewidth=1.6, label='KDE', clip=(-xmax, xmax))
        # Normal curve
        ax.plot(xgrid, normal_pdf, color='#d62728', linewidth=2.0, label=f'Normal fit (μ={mu:.2f} mm, σ={sigma:.2f} mm)')
        ax.set_xlim(-xmax, xmax)
        ax.set_title(f"{sol}: Residuals with fitted normal curve")
        ax.set_xlabel("Detrended residual (mm)")
        ax.set_ylabel("Density")
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        ax.legend()
        # Annotate sample size
        n = x.size
        ax.text(0.99, 0.97, f"n = {n}", transform=ax.transAxes, ha='right', va='top')
        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved printable PDF: {OUTPUT_PDF}")

In [ ]:
df = detrended_df.copy()
df = df.dropna(subset=['station','detrended_mm'])

# Choose which stations to include (e.g., top by sample size)
counts = df.groupby('station').size().sort_values(ascending=False)
stations_to_plot = counts.head(24).index  # change as needed

def fit_gaussian(x, robust=True):
    mu = float(np.mean(x))
    if robust:
        med = np.median(x)
        mad = np.median(np.abs(x - med))
        sigma = 1.4826 * mad
    else:
        sigma = float(np.std(x, ddof=1))
    return mu, max(sigma, 1e-9)

# precompute a large standard normal sample to approximate theoretical quantiles
sim_norm = np.random.normal(size=200000)

OUTPUT_PDF = "station_panels_normal_and_qq_A4.pdf"
sns.set(style="whitegrid")
with PdfPages(OUTPUT_PDF) as pdf:
    for st in stations_to_plot:
        x = df.loc[df['station'] == st, 'detrended_mm'].to_numpy()
        if x.size < 30:
            continue  # skip very small samples
        mu, sigma = fit_gaussian(x, robust=True)
        # Normal PDF grid
        xmax = float(np.nanpercentile(np.abs(x), 99))
        xgrid = np.linspace(-xmax, xmax, 800)
        normal_pdf = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((xgrid - mu)/sigma)**2)

        fig, axes = plt.subplots(1, 2, figsize=(11.7, 8.3))  # A4 landscape

        # --- Left: histogram + KDE + normal overlay ---
        ax = axes[0]
        ax.hist(x, bins=50, density=True, color='#c7e9c0', edgecolor='#238b45', alpha=0.6, label='Histogram (density)')
        sns.kdeplot(x, ax=ax, color='#08519c', linewidth=1.6, label='KDE', clip=(-xmax, xmax))
        ax.plot(xgrid, normal_pdf, color='#d62728', linewidth=2.0, label=f'Normal fit (μ={mu:.2f}, σ={sigma:.2f})')
        ax.set_xlim(-xmax, xmax)
        ax.set_title(f"{st}: Residuals vs normal")
        ax.set_xlabel("Detrended residual (mm)")
        ax.set_ylabel("Density")
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        ax.legend()

        # --- Right: Q‑Q plot against normal ---
        ax = axes[1]
        # Sort data & compute empirical quantiles
        x_sorted = np.sort(x)
        n = x_sorted.size
        probs = (np.arange(1, n+1) - 0.5) / n
        # Approximate theoretical normal quantiles via simulated sample percentiles
        theo = np.percentile(sim_norm, probs*100)
        # Standardise data to unit normal scale before comparing
        z = (x_sorted - mu) / sigma
        ax.scatter(theo, z, s=8, color='#636363', alpha=0.7)
        # 45° reference line
        minv = min(theo.min(), z.min())
        maxv = max(theo.max(), z.max())
        ax.plot([minv, maxv], [minv, maxv], color='black', linewidth=1.2, linestyle='--', label='Ideal (normal)')
        ax.set_title(f"{st}: Q‑Q vs Normal (robust z‑scores)")
        ax.set_xlabel("Theoretical normal quantiles")
        ax.set_ylabel("Empirical quantiles (z)")
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.legend()
        # Sample size
        axes[0].text(0.99, 0.97, f"n = {n}", transform=axes[0].transAxes, ha='right', va='top')

        plt.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)

print(f"Saved printable PDF: {OUTPUT_PDF}")

In [ ]:
# Tidy input
df = detrended_df.copy()
df = df.dropna(subset=['solutiontype', 'detrended_mm'])

# Choose how to centre the residuals: "mean" or "median"
CENTRE = "median"   

# Build offsets per solution type (offset = residual - centre)
curves = []
for sol, g in df.groupby('solutiontype'):
    x = g['detrended_mm'].to_numpy()
    centre = np.mean(x) if CENTRE == "mean" else np.median(x)
    curves.append(pd.DataFrame({'solutiontype': sol, 'offset_mm': x - centre}))

cdf = pd.concat(curves, ignore_index=True).dropna()

# Axis label with British English
x_label = "Mean offset (mm)" if CENTRE == "mean" else "Median offset (mm)"

# Plot: KDE overlays of centred residuals
OUTPUT_PDF = "offset_kde_overlay_A4.pdf"
sns.set(style="whitegrid")
with PdfPages(OUTPUT_PDF) as pdf:
    fig, ax = plt.subplots(figsize=(11.7, 8.3))  # A4 landscape

    # Symmetric limits based on pooled 99th percentile
    xlim = float(np.nanpercentile(np.abs(cdf['offset_mm']), 99))

    # Order legend by sample size (optional)
    sol_order = (cdf.groupby('solutiontype')['offset_mm']
                   .size().sort_values(ascending=False).index)

    for sol in sol_order:
        sns.kdeplot(
            data=cdf[cdf['solutiontype'] == sol],
            x='offset_mm',
            ax=ax, label=sol, linewidth=1.6,
            clip=(-xlim, xlim)
        )

    ax.set_xlim(-xlim, xlim)
    ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)  # zero line (centred)
    ax.set_title(f"Centred residual shapes by solution type (offset from {CENTRE})")
    ax.set_xlabel(x_label)
    ax.set_ylabel("Density")
    ax.grid(True, axis='y', linestyle='--', alpha=0.3)
    ax.legend(title="Solution type", ncol=2)
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

print(f"Saved printable PDF: {OUTPUT_PDF}")

### Producing confrence figures